In [1]:
import pandas as pd

splits = {'train': 'plain_text/train-00000-of-00001.parquet', 'validation': 'plain_text/validation-00000-of-00001.parquet'}
df_squad_train = pd.read_parquet("hf://datasets/rajpurkar/squad/" + splits["train"])
df_squad_test = pd.read_parquet("hf://datasets/rajpurkar/squad/" + splits["validation"])

In [2]:
print(df_squad_train)
print(df_squad_test)

                             id                     title  \
0      5733be284776f41900661182  University_of_Notre_Dame   
1      5733be284776f4190066117f  University_of_Notre_Dame   
2      5733be284776f41900661180  University_of_Notre_Dame   
3      5733be284776f41900661181  University_of_Notre_Dame   
4      5733be284776f4190066117e  University_of_Notre_Dame   
...                         ...                       ...   
87594  5735d259012e2f140011a09d                 Kathmandu   
87595  5735d259012e2f140011a09e                 Kathmandu   
87596  5735d259012e2f140011a09f                 Kathmandu   
87597  5735d259012e2f140011a0a0                 Kathmandu   
87598  5735d259012e2f140011a0a1                 Kathmandu   

                                                 context  \
0      Architecturally, the school has a Catholic cha...   
1      Architecturally, the school has a Catholic cha...   
2      Architecturally, the school has a Catholic cha...   
3      Architecturally, the

In [3]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df_sciq_train = pd.read_parquet("hf://datasets/allenai/sciq/" + splits["train"])
df_sciq_test = pd.read_parquet("hf://datasets/allenai/sciq/" + splits["validation"])


In [4]:
print(df_sciq_train)
print(df_sciq_test)

                                                question      distractor3  \
0      What type of organism is commonly used in prep...          viruses   
1      What phenomenon makes global winds blow northe...  tropical effect   
2      Changes from a less-ordered state to a more-or...      endothermic   
3         What is the least dangerous radioactive decay?       zeta decay   
4      Kilauea in hawaii is the world’s most continuo...            magma   
...                                                  ...              ...   
11674  The enzyme pepsin plays an important role in t...           lipids   
11675  What remains a constant of radioactive substan...          acidity   
11676  Terrestrial ecosystems, also known for their d...       substrates   
11677  High explosives create shock waves that exceed...       turbulence   
11678  What do you call a structure composed of two o...           system   

            distractor1         distractor2        correct_answer  \
0     

In [5]:
import pandas as pd

splits = {'train': 'main/train-00000-of-00001.parquet', 'validation': 'main/validation-00000-of-00001.parquet', 'test': 'main/test-00000-of-00001.parquet'}
df_open_book_train = pd.read_parquet("hf://datasets/allenai/openbookqa/" + splits["train"])
df_open_book_test = pd.read_parquet("hf://datasets/allenai/openbookqa/" + splits["validation"])


In [6]:
# print(df_open_book_train)
# print(df_open_book_test)
print(df_open_book_train.shape)
print(df_open_book_test.shape)

(4957, 4)
(500, 4)


In [7]:
import json

def format_educational_data(df_squad, df_sciq, df_obqa):
    unified_data = []

    # 1. Process SQuAD (Grounded Question Answering)
    # We take the first answer from the 'answers' dictionary column
    for _, row in df_squad.head(3000).iterrows():
        unified_data.append({
            "instruction": "Based on the provided context, answer the student's question accurately.",
            "input": f"Context: {row['context']}\nQuestion: {row['question']}",
            "output": row['answers']['text'][0]
        })

    # 2. Process SciQ (Distractor & MCQ Generation)
    for _, row in df_sciq.head(3000).iterrows():
        # SciQ has distractors and support text
        context = row['support'] if row['support'] else "General Science Knowledge"
        unified_data.append({
            "instruction": "Generate a multiple-choice question with 3 plausible distractors based on this concept.",
            "input": f"Concept: {context}\nTopic: {row['question']}",
            "output": f"Correct Answer: {row['correct_answer']}\nDistractors: {row['distractor1']}, {row['distractor2']}, {row['distractor3']}"
        })

    # 3. Process OpenBookQA (Scientific Reasoning)
    # OpenBookQA has 'choices' as a dict and 'answerKey' as a letter (A, B, C, D)
    for _, row in df_obqa.head(3000).iterrows():
        choices_text = [f"{label}: {text}" for label, text in zip(row['choices']['label'], row['choices']['text'])]
        unified_data.append({
            "instruction": "Analyze the question and identify the correct scientific conclusion from the choices.",
            "input": f"Question: {row['question_stem']}\nChoices: {', '.join(choices_text)}",
            "output": f"The correct answer is {row['answerKey']}."
        })

    return unified_data

# Execute formatting
phase1_list = format_educational_data(df_squad_train, df_sciq_train, df_open_book_train)

In [8]:
output_file = "tutor_train_phase1.jsonl"

with open(output_file, 'w') as f:
    for entry in phase1_list:
        f.write(json.dumps(entry) + '\n')

print(f"✅ Phase 1 Complete! {len(phase1_list)} samples saved to {output_file}")

✅ Phase 1 Complete! 9000 samples saved to tutor_train_phase1.jsonl


In [9]:
from datasets import load_dataset

# Load the file we just created
dataset = load_dataset("json", data_files=output_file, split="train")

# Shuffle to mix SQuAD, SciQ, and OBQA together
dataset = dataset.shuffle(seed=42)

print(dataset)
# View a sample to verify
print("\nSample Data:")
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 9000
})

Sample Data:
{'instruction': "Based on the provided context, answer the student's question accurately.", 'input': "Context: The funeral, held at the Church of the Madeleine in Paris, was delayed almost two weeks, until 30 October. Entrance was restricted to ticket holders as many people were expected to attend. Over 3,000 people arrived without invitations, from as far as London, Berlin and Vienna, and were excluded.\nQuestion: Where was Chopin's funeral held?", 'output': 'Church of the Madeleine'}
